# 🌺 Balinese Voice Cloning + Emotion Profiles — Google Colab

Semua data (voice profile, contoh suara per emosi, hasil profiling prosodi, output, dan cache model)
disimpan di **Google Drive**, jadi saat Colab dijalankan ulang **tidak perlu enroll / profiling ulang**.

```
MyDrive/voice-changer/
├── emotion_samples/<Profile>/<emosi>.wav + <emosi>.txt   ← taruh contoh suara dosen di sini (opsional)
├── voice_profiles/                                     ← profile + emosi + prosody profile (cache)
├── outputs/                                            ← hasil TTS
├── dataset/lora_manifest.jsonl
└── hf_cache/                                           ← model VoxCPM2 (opsional, ±5 GB)
```

**Runtime → Change runtime type → GPU (T4)**, lalu jalankan sel dari atas ke bawah.

In [ ]:
#@title 1. Mount Google Drive & atur folder data
DRIVE_DIR = "/content/drive/MyDrive/voice-changer"  #@param {type:"string"}
CACHE_MODEL_ON_DRIVE = True  #@param {type:"boolean"}

import os
from google.colab import drive

drive.mount("/content/drive")
for sub in ["voice_profiles", "emotion_samples", "outputs", "dataset"]:
    os.makedirs(os.path.join(DRIVE_DIR, sub), exist_ok=True)

os.environ["VOICE_DATA_DIR"] = DRIVE_DIR  # dibaca oleh voice_manager.py & tts_engine.py
if CACHE_MODEL_ON_DRIVE:
    os.environ["HF_HOME"] = os.path.join(DRIVE_DIR, "hf_cache")  # model diunduh sekali saja ke Drive
print("Data folder :", DRIVE_DIR)
print("Model cache :", os.environ.get("HF_HOME", "~/.cache/huggingface (hilang saat runtime reset)"))

In [ ]:
#@title 2. Ambil kode aplikasi
REPO_URL = "https://github.com/gungsatya/voice-changer.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
APP_DIR = "/content/voice-changer"

# Repo privat? Pakai URL dengan token: https://<TOKEN>@github.com/gungsatya/voice-changer.git
# (simpan token di Colab Secrets, jangan ditulis langsung di notebook).
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    !git -C {APP_DIR} fetch -q origin {BRANCH} && git -C {APP_DIR} checkout -q FETCH_HEAD
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {APP_DIR}
!git -C {APP_DIR} log --oneline -1

In [ ]:
#@title 3. Install dependency (torch sudah tersedia di Colab)
!pip install -q "voxcpm>=2.0.3" "gradio>=6,<7" soundfile librosa pandas
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (ganti runtime ke GPU)")

In [ ]:
#@title 4. (Opsional) Profiling contoh suara emosi dari Drive
#@markdown Taruh file di `emotion_samples/<Profile>/<emosi>.wav` + `<emosi>.txt` (transkrip persis).
#@markdown Contoh: `Bli_Made/normal.wav`, `Bli_Made/marah.wav`, `Bli_Made/ceria.wav`.
#@markdown Clip yang sudah pernah diprofiling dan tidak berubah akan **dilewati** (pakai cache di Drive).
!cd {APP_DIR} && python import_emotions.py

In [ ]:
#@title 5. Jalankan aplikasi (buka link *.gradio.live)
!cd {APP_DIR} && python app.py --share

### Cara pakai emosi per kata
1. Tab **1** — enroll speaker (clip ini otomatis menjadi emosi `normal`).
2. Tab **2** — tambahkan clip `marah`, `ceria`, `sedih`, … untuk speaker yang sama (atau pakai sel 4).
3. Tab **3** — pilih mode **Emotion Script**, lalu tulis teks dengan tag:

```
[normal] Om Swastyastu. [marah] Kenapa kamu terlambat lagi? [ceria] Tapi tiang seneng pisan kamu datang!
Tiang [ceria]seneng [normal]pisan.
```

Setiap segmen disintesis dengan clip emosinya sendiri sebagai prompt, sehingga intonasinya mengikuti
emosi tersebut, lalu digabung menjadi satu file.